In [1]:
!pip install -q groq requests beautifulsoup4

In [2]:
import requests
import json
import os
import re

from bs4 import BeautifulSoup
from datetime import datetime
from getpass import getpass
from groq import Groq

GROQ_API_KEY = getpass("Enter your Groq API Key: ")

client = Groq(api_key=GROQ_API_KEY)

available_models = {
    model.id for model in client.models.list().data
}

preferred_models = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant"
]

MODEL_NAME = next(
    (model for model in preferred_models if model in available_models),
    None
)

if MODEL_NAME is None:
    raise RuntimeError(
        "No supported model found. Check Groq Console."
    )

STORAGE_FILE = "summaries.json"

print("Groq connected successfully!")
print("Selected model:", MODEL_NAME)
print("Kata 4 configuration completed!")

Enter your Groq API Key: ··········
Groq connected successfully!
Selected model: openai/gpt-oss-120b
Kata 4 configuration completed!


In [3]:
class ArticleExtractor:

    def extract(self, url):


        if not url.startswith(("http://", "https://")):
            raise ValueError("Please provide a valid HTTP/HTTPS URL.")

        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
            )
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=25
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )


        title_tag = (
            soup.find("meta", property="og:title")
            or soup.find("meta", attrs={"name": "twitter:title"})
        )

        if title_tag:
            title = title_tag.get("content", "")
        elif soup.title:
            title = soup.title.get_text(strip=True)
        else:
            title = "Untitled Article"

        date_tag = (
            soup.find("meta", property="article:published_time")
            or soup.find("meta", attrs={"name": "date"})
            or soup.find("meta", attrs={"name": "pubdate"})
        )

        published_date = (
            date_tag.get("content", "Unknown")
            if date_tag else "Unknown"
        )

        for tag in soup([
            "script", "style", "nav",
            "footer", "header", "aside",
            "form", "noscript"
        ]):
            tag.decompose()


        article = soup.find("article")

        if article is None:
            article = soup.body or soup

        paragraphs = article.find_all("p")

        content = "\n".join(
            p.get_text(" ", strip=True)
            for p in paragraphs
            if len(p.get_text(strip=True)) > 40
        )

        if not content.strip():
            raise ValueError(
                "Unable to extract article text from this webpage."
            )

        content = " ".join(content.split()[:2000])

        return {
            "url": url,
            "title": title,
            "date": published_date,
            "content": content
        }


extractor = ArticleExtractor()

print("ArticleExtractor initialized successfully!")

ArticleExtractor initialized successfully!


In [4]:
class NewsSummarizer:

    def summarize(self, article):

        prompt = f"""
You are an expert news summarization assistant.

Analyze this news article.

TITLE:
{article['title']}

PUBLICATION DATE:
{article['date']}

ARTICLE CONTENT:
{article['content']}

TASK:
Generate a concise and informative summary.

Return ONLY a valid JSON object with these fields:

{{
    "date": "publication date",
    "summary": "concise summary of the article",
    "articleType": "Technology, Finance, Healthcare,
                    Politics, Business or Other"
}}

RULES:
1. Include only important information.
2. Do not invent facts.
3. Preserve important names and figures.
4. Use the article content as the only source.
5. Return valid JSON without Markdown.
"""

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a professional news summarizer. "
                        "Return structured JSON only."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_completion_tokens=2048
        )

        output = response.choices[0].message.content.strip()

        output = re.sub(
            r"^```(?:json)?\s*|\s*```$",
            "",
            output,
            flags=re.IGNORECASE
        ).strip()

        try:
            result = json.loads(output)
        except json.JSONDecodeError:
            raise ValueError(
                "The LLM did not return valid JSON. Try again."
            )

        required_fields = [
            "date", "summary", "articleType"
        ]

        if not all(
            field in result for field in required_fields
        ):
            raise ValueError("Missing required summary fields.")


        result["title"] = article["title"]
        result["url"] = article["url"]

        return result


summarizer = NewsSummarizer()

print("NewsSummarizer initialized successfully!")

NewsSummarizer initialized successfully!


In [5]:
class ArticleStorage:

    def __init__(self, filename=STORAGE_FILE):
        self.filename = filename

    def load_summaries(self):

        if not os.path.exists(self.filename):
            return []

        with open(
            self.filename,
            "r",
            encoding="utf-8"
        ) as file:

            data = json.load(file)

        if not isinstance(data, list):
            raise ValueError("Invalid summaries.json format.")

        return data

    def save_summary(self, summary):

        summaries = self.load_summaries()


        for item in summaries:

            if item.get("url") == summary.get("url"):
                print("Article already exists in storage!")
                return

        summaries.append(summary)

        with open(
            self.filename,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                summaries,
                file,
                indent=4,
                ensure_ascii=False
            )

        print("Summary saved successfully!")

    def display_summaries(self):

        summaries = self.load_summaries()

        if not summaries:
            print("No summaries available.")
            return

        for index, item in enumerate(summaries, start=1):

            print(f"\nArticle {index}")
            print("Title:", item["title"])
            print("Date:", item["date"])
            print("Category:", item["articleType"])
            print("Summary:", item["summary"])
            print("URL:", item["url"])


storage = ArticleStorage()

print("ArticleStorage initialized successfully!")

ArticleStorage initialized successfully!


In [6]:
class SummaryQAEngine:

    def retrieve_relevant_summaries(
        self,
        question,
        summaries,
        top_k=3
    ):


        stop_words = {
            "what", "when", "where", "which",
            "who", "how", "why", "the", "is",
            "are", "was", "were", "and", "for",
            "about", "this", "that", "from"
        }

        question_words = set(
            re.findall(r"\b[a-zA-Z]{3,}\b", question.lower())
        ) - stop_words

        scored_summaries = []

        for item in summaries:

            text = (
                item.get("title", "") + " " +
                item.get("summary", "") + " " +
                item.get("articleType", "")
            ).lower()

            document_words = set(
                re.findall(r"\b[a-zA-Z]{3,}\b", text)
            )

            score = len(
                question_words.intersection(document_words)
            )

            if score > 0:
                scored_summaries.append((score, item))

        scored_summaries.sort(
            key=lambda x: x[0],
            reverse=True
        )

        if not scored_summaries:
            return summaries[-top_k:]

        return [
            item
            for score, item in scored_summaries[:top_k]
        ]

    def answer_question(self, question):

        summaries = storage.load_summaries()

        if not summaries:
            return "No saved news summaries available."

        relevant_summaries = (
            self.retrieve_relevant_summaries(
                question,
                summaries
            )
        )

        context = json.dumps(
            relevant_summaries,
            indent=2,
            ensure_ascii=False
        )

        prompt = f"""
You are a news question-answering assistant.

Use ONLY the following saved news summaries.

NEWS CONTEXT:
{context}

USER QUESTION:
{question}

INSTRUCTIONS:
1. Answer using the provided news context.
2. Do not invent information.
3. If the answer is unavailable, clearly say so.
4. Keep the answer clear and concise.
5. Mention the relevant article title when possible.
"""

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You answer questions using saved news summaries only."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_completion_tokens=1024
        )

        return response.choices[0].message.content.strip()


qa_engine = SummaryQAEngine()

print("SummaryQAEngine initialized successfully!")

SummaryQAEngine initialized successfully!


In [7]:
def process_news_article(url):

    try:
        # Extract article
        print("\nStep 1: Extracting article...")

        article = extractor.extract(url)

        print("Title:", article["title"])
        print("Date:", article["date"])

        # Generate summary
        print("\nStep 2: Generating summary...")

        summary = summarizer.summarize(article)

        print("\nGenerated Summary:")
        print(json.dumps(summary, indent=4))

        # Save summary
        print("\nStep 3: Saving summary...")

        storage.save_summary(summary)

        return summary

    except Exception as error:

        print("\nError processing article:", error)

        return None



while True:

    news_url = input(
        "\nEnter a news article URL (or type exit): "
    ).strip()

    if news_url.lower() == "exit":
        break

    process_news_article(news_url)



print("\n========== SAVED NEWS SUMMARIES ==========")

storage.display_summaries()



print("\n========== NEWS QUESTION ANSWERING ==========")

while True:

    question = input(
        "\nAsk a question (or type exit): "
    ).strip()

    if question.lower() == "exit":
        break

    if not question:
        continue

    answer = qa_engine.answer_question(question)

    print("\nAnswer:")
    print(answer)


Enter a news article URL (or type exit): https://www.bbc.com/news/articles/c0qj4z8e2y7o

Step 1: Extracting article...

Error processing article: 404 Client Error: Not Found for url: https://www.bbc.com/news/articles/c0qj4z8e2y7o

Enter a news article URL (or type exit): https://www.bbc.com/news/articles/c3rryrryk59zo

Step 1: Extracting article...
Title: Three Iran-backed suspects plotted attacks including RAF Fairford incident, US says
Date: Unknown

Step 2: Generating summary...

Generated Summary:
{
    "date": "Unknown",
    "summary": "The U.S. Department of Justice unsealed an eight\u2011count indictment charging three Iran\u2011backed operatives\u2014Mohsen Mohajeri, Mohammad Ali al\u2011Hassani, and Mohammad Baqer Saad Dawood al\u2011Saadi\u2014with planning and supporting a series of terrorist attacks in Europe and the United States. The indictment links them to the September 27 incident near the UK\u2019s RAF Fairford base, where suspicious vans carrying fuel were discovere